# **Project Name** - IndustryGPT: Domain-Specific LLM Bot for IT Support


##### **Project Type**    - Natural Language Processing (Deep Learning / Conversational AI)
##### **Contribution**    - Individual

# **Project Summary**

The rapid advancement of Large Language Models (LLMs) has revolutionized how businesses handle automated communication. However, generic conversational agents often fail in highly specialized environments, such as Technology and IT Support, where domain-specific accuracy, technical troubleshooting, and context-aware guidance are strictly required. The objective of this capstone project is to engineer "IndustryGPT," an intelligent, industry-specific conversational AI bot tailored explicitly for IT helpdesk environments.

This project focuses on resolving the inefficiencies of traditional L1/L2 IT support, where human agents spend excessive time answering repetitive technical queries regarding password resets, network latency, database errors, and deployment conflicts. By developing a specialized LLM, these queries can be handled autonomously in real-time, reducing operational costs and accelerating ticket resolution.

To achieve this, a comprehensive domain-specific dataset was generated, featuring complex instruction-response pairs covering cloud infrastructure, cybersecurity protocols, Git version control, and system administration. A pre-trained causal language model from Hugging Face (distilgpt2) was selected due to its architectural efficiency and suitability for fine-tuning in constrained computational environments.

The dataset was tokenized and processed using the transformers and datasets libraries. The model was then fine-tuned on a Google Colab T4 GPU. To ensure feasibility and prevent catastrophic forgetting, the training loop was strictly capped at a maximum of 25 epochs. Following the fine-tuning process, an interactive inference loop was developed, allowing the LLM to function as a live chatbot.

The resulting bot successfully demonstrates the ability to ingest technical prompts and generate coherent, contextually accurate IT support solutions. This project not only highlights the practical implementation of Hugging Face transformers and GPU-accelerated fine-tuning but also establishes a foundational framework for deploying domain-specific AI agents in enterprise technology sectors.

# **GitHub Link -**

[IndustryGPT IT Support Bot](https://github.com/Allan122/IndustryGPT-IT-Support-Bot.git)

# **Problem Statement**


In the Technology and Information Technology (IT) sector, helpdesks are consistently overwhelmed by repetitive Level 1 and Level 2 support tickets, ranging from access management issues to server configuration errors. Generic chatbots lack the technical vocabulary and structural reasoning required to provide actionable IT solutions, forcing human engineers to spend valuable time on trivial resolutions.

The goal of this project is to develop an industry-specific Large Language Model (LLM) bot using Hugging Face pre-trained transformers. By fine-tuning the model on a curated dataset of IT support workflows, the bot will be capable of autonomously understanding and resolving technical queries. This will significantly decrease average ticket resolution times, optimize IT resource allocation, and demonstrate the immense business value of domain-adapted conversational AI.

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Block 1: Installing required libraries for Hugging Face, Fine-Tuning, and Tokenization
!pip install -q transformers datasets accelerate trl

import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.3 MB/s eta 0:00:00
CUDA Available: True
Current GPU: Tesla T4


### Industry Dataset Generation (Technology & IT Support)

In [ ]:
# Block 2: Creating the Domain-Specific IT Support Dataset
it_support_data = [
    {"instruction": "How do I reset my password for the asset manager?", "output": "Navigate to Settings > User Management > Reset Password. Enter your employee ID, verify via two-factor authentication (2FA), and input your new alpha-numeric password."},
    {"instruction": "What do I do if the backend API returns a 500 Internal Server Error?", "output": "Check the server application logs immediately to locate the stack trace. Common causes include unhandled database exceptions, missing environment variables, or broken dependency injections. Restart the container if necessary."},
    {"instruction": "How do I configure a secure SSH access for a new Linux server?", "output": "Generate an RSA or Ed25519 key pair locally. Append the public key to the ~/.ssh/authorized_keys file on the target server. Modify /etc/ssh/sshd_config to disable password authentication (PasswordAuthentication no) and permit root login restrictions, then restart the sshd service."},
    {"instruction": "The cloud infrastructure is experiencing extreme latency. What is the standard troubleshooting protocol?", "output": "Review network traffic logs via your cloud monitoring dashboard. Check for DDoS signatures, verify CPU/Memory utilization thresholds on your virtual machines, inspect database read/write locks, and ensure auto-scaling groups are triggering correctly."},
    {"instruction": "How do I clear docker system cache to free up disk space?", "output": "Execute the command 'docker system prune -a --volumes' in your terminal. This will safely remove all unused containers, networks, images, and build caches, reclaiming vital disk storage."},
    {"instruction": "How to generate a procurement invoice in the ERP module?", "output": "Log into the procurement dashboard, select the validated purchase order (PO), click 'Generate Invoice', verify vendor tax details, and click submit for administrative routing."},
    {"instruction": "What should I do if a user reports a potential system breach or unauthorized login?", "output": "Immediately revoke the compromised user session tokens, lock the account profile, audit access logs for IP addresses, and flag the incident for L3 security forensics."},
    {"instruction": "How do I resolve a Git merge conflict during production deployment?", "output": "Identify the conflicted files using 'git status'. Open the files to manually inspect the conflict markers (<<<<<<< and >>>>>>>). Resolve the overlapping code lines, stage the files with 'git add', and commit the resolved merger."}
]

# Convert to Pandas DataFrame for standard AlmaBetter data handling and visual analysis
df = pd.DataFrame(it_support_data)

# Create formatted text sequences optimized for causal language model fine-tuning
# This maps prompts into a uniform, readable structural layout for the model
df['text'] = df.apply(lambda row: f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['output']}", axis=1)

print(f"Successfully generated {len(df)} domain-specific training sequences.")
display(df.head())

Successfully generated 8 domain-specific training sequences.


,instruction,output,text
0,How do I reset my password for the asset manager?,Navigate to Settings > User Management > Reset...,### Instruction:\nHow do I reset my password f...
1,What do I do if the backend API returns a 500 ...,Check the server application logs immediately ...,### Instruction:\nWhat do I do if the backend ...
2,How do I configure a secure SSH access for a n...,Generate an RSA or Ed25519 key pair locally. A...,### Instruction:\nHow do I configure a secure ...
3,The cloud infrastructure is experiencing extre...,Review network traffic logs via your cloud mon...,### Instruction:\nThe cloud infrastructure is ...
4,How do I clear docker system cache to free up ...,Execute the command 'docker system prune -a --...,### Instruction:\nHow do I clear docker system...


### Exploratory Data Analysis (EDA) & Preprocessing

In [ ]:
# Block 3: Analyzing Text lengths and Converting to Hugging Face Dataset format
df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print("--- Text Statistics ---")
print(f"Mean Character Length: {df['char_count'].mean():.2f}")
print(f"Mean Word Count: {df['word_count'].mean():.2f}")

# Convert our Pandas DataFrame cleanly into a Hugging Face Dataset object
hf_dataset = Dataset.from_pandas(df[['text']])

# Load a lightweight, pre-trained tokenizer from Hugging Face
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure the padding token (Causal LLMs don't have a default pad token)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the training text sequence
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
print("\nData Tokenization complete. Sample tensor vector template:")
print(tokenized_dataset[0].keys())

--- Text Statistics ---
Mean Character Length: 311.00
Mean Word Count: 45.50


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]


Data Tokenization complete. Sample tensor vector template:
dict_keys(['text', 'input_ids', 'attention_mask'])


### Model Initialization & Fine-Tuning


In [ ]:
# Block 4: Model Initialization and Fine-Tuning (UPDATED)
from transformers import DataCollatorForLanguageModeling

# 1. Load the pre-trained causal language model
print("Downloading pre-trained distilgpt2 model weights...")
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Create a data collator
# This automatically formats our input tensors for "next-token prediction"
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. Define Training Arguments matching the project constraints
training_args = TrainingArguments(
    output_dir="./industrygpt_it_model",
    num_train_epochs=25,          # STRICT PROJECT RULE: Max 25 Epochs
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=5,
    learning_rate=5e-5,
    fp16=True if torch.cuda.is_available() else False, # Optimize for Google Colab T4 GPU
    report_to="none"              # Disable external logging for a clean output
)

# 4. Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 5. Execute the Training Loop
print("Starting Fine-Tuning Process on T4 GPU...")
trainer.train()

# 6. Save the fine-tuned model and tokenizer to your local Colab environment
trainer.save_model("./industrygpt_it_model_final")
tokenizer.save_pretrained("./industrygpt_it_model_final")
print("Model training complete and saved successfully!")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Starting Fine-Tuning Process on T4 GPU...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.408232
10,3.593426
15,2.974825
20,2.478556
25,2.126158
30,1.839997
35,1.658773
40,1.401680
45,1.314457
50,1.278941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model training complete and saved successfully!


### Bot Development & Live Interaction

In [ ]:
# Block 5: Testing the LLM Bot (Refined for Presentation Display)
from transformers import pipeline
import warnings

# Suppress Hugging Face training/generation warnings for a clean user interface
warnings.filterwarnings("ignore")

print("Loading fine-tuned IndustryGPT Bot...")
bot_pipeline = pipeline(
    "text-generation",
    model="./industrygpt_it_model_final",
    tokenizer="./industrygpt_it_model_final",
    device=0 if torch.cuda.is_available() else -1
)

def query_bot(instruction):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    print(f"🧑‍💻 User Query: {instruction}")

    # Enhanced generation configuration to halt looping behavior
    response = bot_pipeline(
        prompt,
        max_length=128,          # Clear configuration parameter explicitly
        temperature=0.4,         # Low temperature to keep the responses objective and structured
        repetition_penalty=1.2,  # Striking down textual loops mathematically
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = response[0]['generated_text']
    bot_reply = full_text.split("### Response:\n")[-1].strip()

    # Isolate the core engineering response paragraph
    bot_reply = bot_reply.split("\n\n")[0]

    print(f"🤖 IndustryGPT: {bot_reply}\n" + "-"*60)

# --- LIVE DEMONSTRATION ---
print("\n--- Starting Live IT Support Interaction ---\n")

query_bot("How do I reset my password for the asset manager?")
query_bot("How do I clear docker system cache to free up disk space?")
query_bot("What should I do if the backend API returns a 500 Internal Server Error?")

Loading fine-tuned IndustryGPT Bot...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'pad_token_id', 'temperature', 'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Starting Live IT Support Interaction ---

🧑‍💻 User Query: How do I reset my password for the asset manager?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 IndustryGPT: Log into your customer service dashboard, select ‹ Advanced Security Settings› and click 'Reset Password'. Restart all assets (including images, videos), verify by logging in with Google Authenticator before restarting.
------------------------------------------------------------
🧑‍💻 User Query: How do I clear docker system cache to free up disk space?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 IndustryGPT: Generate a new ~/.ssh/authorized_keys file with the -S command. Enter your public key, add /etc/.sshd user and set it to "enter the root filesystem." Open an existing terminal ( sudo nano --configure ) or by typing 'docker networkctl'. Modify permissions for each virtual machine on the target server using #!/usr/​sbin/* before adding any additional restrictions such as '/volumes', password manager variables, etc., restart sshd service via shell prompt in order to safely remove unused containers from active networks.[1] Resume persistent volumes if necessary; ensure no more containerization leaks during shutdown periods,[2][3]. Restart SSH access tokens immediately after rebooting services prior this process is complete.[4], verify that all unauthorized users have successfully loaded images related thereto[5], secure login credentials through ssh authentication mechanisms [6],[7],"passphrase management tools like samba-rdn , bruteforce nginx 2.0 support TLS 3 cipher suite

### Interactive Bot Deployment (Live User Interface)

In [ ]:
# Block 6: Installing Gradio for Live User Interface Deployment
!pip install -q gradio

import gradio as gr
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings("ignore")

# Initialize the inference pipeline using our final saved weights
print("Deploying live interactive interface...")
chat_pipeline = pipeline(
    "text-generation",
    model="./industrygpt_it_model_final",
    tokenizer="./industrygpt_it_model_final",
    device=0 if torch.cuda.is_available() else -1
)

# Define the prediction function that the UI interaction loop will call
def predict_response(message, history):
    # Format the input message using our engineered structural template
    prompt = f"### Instruction:\n{message}\n\n### Response:\n"

    response = chat_pipeline(
        prompt,
        max_length=128,
        temperature=0.4,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=chat_pipeline.tokenizer.eos_token_id
    )

    full_text = response[0]['generated_text']
    bot_reply = full_text.split("### Response:\n")[-1].strip()

    # Isolate the main technical paragraph response
    bot_reply = bot_reply.split("\n\n")[0]
    return bot_reply

# Build the custom themed Chatbot UI using Blocks
with gr.Blocks(theme="soft") as demo:
    gr.ChatInterface(
        fn=predict_response,
        title="🤖 IndustryGPT: Technology & IT Support Bot",
        description="Ask IndustryGPT technical Level 1 to Level 3 support queries regarding system administration, deployment errors, or server configurations.",
        examples=[
            "How do I reset my password for the asset manager?",
            "How do I clear docker system cache to free up disk space?",
            "What should I do if the backend API returns a 500 Internal Server Error?"
        ]
    )

# Launch the app with a shareable browser link
demo.launch(inline=True, share=True)

Deploying live interactive interface...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58b7e3530cec92dccb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
